# Lab 13: Information Retrieval for RAG

## 1. What is Information Retrieval?

**Information retrieval (IR)** means finding relevant information from a collection of documents based on a user query.

**Example:** A student asks, *"How do transformers process words?"* The retrieval system searches course notes and returns the most relevant passages.

**Simple process:**

`Documents → Numerical representation → Compare with query → Rank → Top results`

## 2. Create a small document collection

Imagine these are short notes taken from an NLP course. Each string represents one document.

In [1]:
documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."]

for i, document in enumerate(documents):
    print(f"Document {i}: {document}")

Document 0: Transformers use self-attention to understand relationships between words.
Document 1: TF-IDF represents documents using word importance scores.
Document 2: Word2Vec learns dense vector representations of words.
Document 3: Named entity recognition identifies names, places, and organizations in text.
Document 4: Information retrieval finds relevant documents for a user query.
Document 5: Cosine similarity measures how similar two text vectors are.


## 3. Convert documents into TF-IDF vectors (sparse retrieval)

A computer cannot compare raw text directly using vector similarity. **TF-IDF** converts each document into a numerical vector.

- **TF** measures how frequently a word occurs in a document.
- **IDF** gives more importance to words that appear in fewer documents.

TF-IDF is a form of **sparse retrieval**: most word positions in the vectors are zero.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", document_vectors.shape)
print("First document vector:", document_vectors[0].toarray().round(2))

TF-IDF matrix shape: (6, 39)
First document vector: [[0.39 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.39 0.   0.   0.   0.   0.   0.39 0.   0.   0.
  0.   0.39 0.39 0.39 0.   0.   0.   0.   0.   0.   0.32]]


**Check:** The matrix has one row per document and one column per vocabulary word. A zero means that word has no TF-IDF weight in that document.

## 4. Search using a query

We use the **same TF-IDF vectorizer** to represent the user's query. Then we calculate **cosine similarity** between the query and every document.

- Higher similarity → more relevant according to TF-IDF.
- A similarity of 0 means the vectors do not share weighted terms.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

query = "How do transformers understand words?"
query_vector = vectorizer.transform([query])
scores = cosine_similarity(query_vector, document_vectors)[0]

for i, score in enumerate(scores):
    print(f"Document {i}: {score:.3f}")

Document 0: 0.633
Document 1: 0.000
Document 2: 0.173
Document 3: 0.000
Document 4: 0.000
Document 5: 0.000


## 5. Retrieve the top-k documents

Rather than reading every document, the system returns the **k** documents with the highest similarity scores.

In [4]:
k = 3
ranked_ids = scores.argsort()[::-1][:k]

for rank, i in enumerate(ranked_ids, start=1):
    print(f"Rank {rank} | Score: {scores[i]:.3f}")
    print(documents[i])
    print()

Rank 1 | Score: 0.633
Transformers use self-attention to understand relationships between words.

Rank 2 | Score: 0.173
Word2Vec learns dense vector representations of words.

Rank 3 | Score: 0.000
Information retrieval finds relevant documents for a user query.



## 6. Student exercises

**Exercise 1 — Change the query**  
Use `search()` to find notes about **named entity recognition**. Which document is ranked first?

**Exercise 2 — Change k**  
Search for `"document representation"` using `k=1` and then `k=3`. What changes?


### Solution

The exercises use a `search()` function. It puts the steps from sections 4 and 5 together: transform the query with the **same** vectorizer, compute cosine similarity with every document, sort the scores and return the top **k** documents.

In [5]:
def search(query, k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    print(f"Query: {query!r} (k={k})")
    for rank, i in enumerate(top_ids, start=1):
        print(f"Rank {rank} | Score: {scores[i]:.3f} | Document {i}: {documents[i]}")
    print()

# Test it with the query from section 4: the result must be the same as above
search("How do transformers understand words?", k=3)

Query: 'How do transformers understand words?' (k=3)
Rank 1 | Score: 0.633 | Document 0: Transformers use self-attention to understand relationships between words.
Rank 2 | Score: 0.173 | Document 2: Word2Vec learns dense vector representations of words.
Rank 3 | Score: 0.000 | Document 4: Information retrieval finds relevant documents for a user query.



#### Exercise 1 — Change the query

Search for notes about **named entity recognition**.

In [6]:
search("named entity recognition", k=3)

Query: 'named entity recognition' (k=3)
Rank 1 | Score: 0.625 | Document 3: Named entity recognition identifies names, places, and organizations in text.
Rank 2 | Score: 0.000 | Document 5: Cosine similarity measures how similar two text vectors are.
Rank 3 | Score: 0.000 | Document 4: Information retrieval finds relevant documents for a user query.



**Answer:** **Document 3** — *"Named entity recognition identifies names, places, and organizations in text."* — is ranked first with a score of about **0.625**.

All three query words (*named*, *entity*, *recognition*) appear only in Document 3, so it is the only document that shares weighted terms with the query. The other documents get a score of **0.000**. They still appear at ranks 2 and 3 only because we asked for `k=3` results, not because they are relevant.

#### Exercise 2 — Change k

Search for `"document representation"` with `k=1` and then with `k=3`.

In [7]:
search("document representation", k=1)
search("document representation", k=3)

Query: 'document representation' (k=1)
Rank 1 | Score: 0.000 | Document 5: Cosine similarity measures how similar two text vectors are.

Query: 'document representation' (k=3)
Rank 1 | Score: 0.000 | Document 5: Cosine similarity measures how similar two text vectors are.
Rank 2 | Score: 0.000 | Document 4: Information retrieval finds relevant documents for a user query.
Rank 3 | Score: 0.000 | Document 3: Named entity recognition identifies names, places, and organizations in text.



**What changes:** `k` only controls **how many** documents are returned. With `k=1` we get one document and with `k=3` we get three. The ranking itself does not change: the first result of `k=3` is the same document as the `k=1` result, and the next documents are simply added below it.

**Important observation:** all the scores are **0.000**, so none of these results is really relevant. TF-IDF matches **exact words** and does no stemming:

- the query word *document* is not in the vocabulary — only *documents* is (Documents 1 and 4);
- *representation* is not in the vocabulary — only *representations* (Document 2) and *represents* (Document 1).

So the query vector is all zeros, every document gets a score of 0, and the "top-k" documents are just picked by the order of the sorting (ties), not by relevance. We can check this:

In [8]:
vocabulary = vectorizer.get_feature_names_out()
for word in ["document", "documents", "representation", "representations", "represents"]:
    print(f"{word:16} in vocabulary: {word in vocabulary}")

print()
print("Query vector non-zero values:", vectorizer.transform(["document representation"]).nnz)

document         in vocabulary: False
documents        in vocabulary: True
representation   in vocabulary: False
representations  in vocabulary: True
represents       in vocabulary: True

Query vector non-zero values: 0


If we use the same word forms as the documents, the search finds the relevant notes. Document 1 (*TF-IDF represents **documents** ...*), Document 2 (*... dense vector **representations** ...*) and Document 4 (*... finds relevant **documents** ...*) now get positive scores:

In [9]:
search("documents representations", k=1)
search("documents representations", k=3)

Query: 'documents representations' (k=1)


Rank 1 | Score: 0.325 | Document 2: Word2Vec learns dense vector representations of words.

Query: 'documents representations' (k=3)
Rank 1 | Score: 0.325 | Document 2: Word2Vec learns dense vector representations of words.
Rank 2 | Score: 0.201 | Document 4: Information retrieval finds relevant documents for a user query.
Rank 3 | Score: 0.188 | Document 1: TF-IDF represents documents using word importance scores.



**Conclusion:** a larger `k` returns more documents (more context for a RAG model, but also more chance of including irrelevant text), and a smaller `k` returns only the best match. In both cases we should also look at the **score**: a document with score 0 is not a real match, even if it appears in the top-k. This word-matching limitation of sparse retrieval is studied in Part 2.

## Summary

You built a small **information retrieval** system using TF-IDF and cosine similarity. It converts documents and queries into vectors, ranks documents by relevance, and returns the top results.

**Connection to RAG:** These retrieved documents can later be passed to a language model as supporting context for answer generation.

__________________________________________________________________________________________

# What I Learned from Lab 13 (Part 1)

In this lab I built a small **information retrieval (IR)** system, which is the "retrieval" step of **RAG** (Retrieval-Augmented Generation).

### The retrieval pipeline
`Documents → TF-IDF vectors → transform the query with the same vectorizer → cosine similarity → rank → top-k`

- **TF-IDF** turns every document into a vector with one value per vocabulary word. Our 6 documents give a **6 × 39** matrix, and most values are zero, so it is a **sparse** representation. `stop_words="english"` removes common words like *how*, *do* and *the*.
- The query must be transformed with the **same fitted vectorizer** (`transform`, not `fit_transform`), so that the query and the documents use the same vocabulary and IDF weights.
- **Cosine similarity** compares the query vector with each document vector. A higher score means more shared important words; 0 means no shared words.
- `scores.argsort()[::-1][:k]` sorts the documents from the highest to the lowest score and keeps the top **k**.

### Results of the exercises
- *"How do transformers understand words?"* → Document 0 (Transformers / self-attention) is first with **0.633**.
- *"named entity recognition"* → Document 3 is first with **0.625**; all other documents score 0.
- Changing **k** changes only the number of returned documents, not the order.
- *"document representation"* scores **0 for every document**, because TF-IDF matches exact words: *document ≠ documents* and *representation ≠ representations*. The top-k list is still filled, so a high rank does not always mean a good match — we must check the score as well.

### Connection to RAG
The top-k retrieved documents are given to a language model as context, so it can generate an answer based on them. Good retrieval is important: if the retrieved documents are wrong, the generated answer will also be wrong.